In [10]:
%pip uninstall -y langchain langchain-community langchain-core langchain-openai
%pip install langchain==0.1.0 langchain-community langchain-openai

Found existing installation: langchain 1.2.10
Uninstalling langchain-1.2.10:
  Successfully uninstalled langchain-1.2.10
Found existing installation: langchain-community 0.4.1
Uninstalling langchain-community-0.4.1:
  Successfully uninstalled langchain-community-0.4.1
Found existing installation: langchain-core 1.2.17
Uninstalling langchain-core-1.2.17:
  Successfully uninstalled langchain-core-1.2.17
Found existing installation: langchain-openai 1.1.10
Uninstalling langchain-openai-1.1.10:
  Successfully uninstalled langchain-openai-1.1.10
Note: you may need to restart the kernel to use updated packages.
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_openai-1.1.10-py3-none-any.whl.metadata (3.1 kB)
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-community to determi

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-chroma 0.2.2 requires langchain-core!=0.3.0,!=0.3.1,!=0.3.10,!=0.3.11,!=0.3.12,!=0.3.13,!=0.3.14,!=0.3.2,!=0.3.3,!=0.3.4,!=0.3.5,!=0.3.6,!=0.3.7,!=0.3.8,!=0.3.9,<0.4.0,>=0.2.43, but you have langchain-core 0.1.23 which is incompatible.
langchain-classic 1.0.2 requires langchain-core<2.0.0,>=1.2.17, but you have langchain-core 0.1.23 which is incompatible.
langchain-classic 1.0.2 requires langsmith<1.0.0,>=0.1.17, but you have langsmith 0.0.87 which is incompatible.
langchain-google-genai 4.2.0 requires langchain-core<2.0.0,>=1.2.5, but you have langchain-core 0.1.23 which is incompatible.
langchain-postgres 0.0.13 requires langchain-core<0.4.0,>=0.2.13, but you have langchain-core 0.1.23 which is incompatible.
langchain-text-splitters 1.1.1 requires langchain-core<2.0.0,>=1.2.13, but you have langchain-c

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [2]:
sample_text = """
LangChain is an open-source framework for building applications with large language models (LLMs).
It allows developers to combine LLMs with external data sources, APIs, and custom logic.
Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval and LLMs
to answer questions with up-to-date, domain-specific, or long-tail knowledge.

Prashant Nair is one of the best trainers of AI.
"""
with open("sample.txt", "w") as f:
    f.write(sample_text)

In [3]:
!pip install -r requirements.txt

In [6]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Step1:Load file
loader = TextLoader("sample.txt")
docs = loader.load()

# Step2:Chunk (for demo, small size)
splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10)
chunks = splitter.split_documents(docs)
print(f"Chunks: {len(chunks)}")
for i, c in enumerate(chunks):
    print(f"Chunk {i+1}: {c.page_content}")

Chunks: 12
Chunk 1: LangChain is an open-source framework for
Chunk 2: for building applications with large language
Chunk 3: language models (LLMs).
Chunk 4: It allows developers to combine LLMs with
Chunk 5: LLMs with external data sources, APIs, and custom
Chunk 6: custom logic.
Chunk 7: Retrieval-Augmented Generation (RAG) is a
Chunk 8: is a technique that combines information
Chunk 9: retrieval and LLMs
Chunk 10: to answer questions with up-to-date,
Chunk 11: domain-specific, or long-tail knowledge.
Chunk 12: Prashant Nair is one of the best trainers of AI.


In [7]:
#Step3: initialize embeddings

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [8]:
#Step4: Initialize Vector Store

from langchain_community.vectorstores import FAISS

#Step5: Build FAISS vector from chunks (with OpenAI or HF embeddings)
vectorstore = FAISS.from_documents(chunks, embeddings)

# Save and reload demo (optional)
vectorstore.save_local("faiss_index")
vectorstore = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)


In [9]:
from langchain_community.vectorstores import Chroma

vectorstore_chroma = Chroma.from_documents(
     chunks,
     embeddings,
     persist_directory="chroma_index"
 )
vectorstore_chroma.persist()
# # To reload:
vectorstore_chroma = Chroma(
     persist_directory="chroma_index",
     embedding_function=embeddings
 )


C:\Users\adira\AppData\Local\Temp\ipykernel_6148\3279036210.py:8: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore_chroma.persist()
C:\Users\adira\AppData\Local\Temp\ipykernel_6148\3279036210.py:10: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore_chroma = Chroma(


In [10]:
query = "What is RAG in LangChain?"
# Using FAISS
results = vectorstore.similarity_search(query, k=3)
print("\n--- FAISS Results ---")
for doc in results:
    print(doc.page_content)

# Using Chroma
results_chroma = vectorstore_chroma.similarity_search(query, k=2)
print("\n--- Chroma Results ---")
for doc in results_chroma:
    print(doc.page_content)



--- FAISS Results ---
LangChain is an open-source framework for
Retrieval-Augmented Generation (RAG) is a
for building applications with large language

--- Chroma Results ---
LangChain is an open-source framework for
LangChain is an open-source framework for


In [11]:
#Retrieve top 3 chunks
retriever = vectorstore.as_retriever(search_kwargs={"k" : 3})

In [12]:
from langchain_openai import ChatOpenAI
from langchain.chains.retrieval import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import PromptTemplate

prompt_template = """You are a teacher teaching AI to students.

{context}

Question: {input}
Answer here:"""
PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "input"]
)


llm = ChatOpenAI(model="gpt-4o-mini")

combine_docs_chain = create_stuff_documents_chain(llm, PROMPT) # PROMPT | llm

qa = create_retrieval_chain(
    retriever=retriever,
    combine_docs_chain=combine_docs_chain
)

result = qa.invoke({"input": "What is Langchain??"})
print(result)

{'input': 'What is Langchain??', 'context': [Document(id='834d1023-e6de-4f54-801d-095d1d2e95d6', metadata={'source': 'sample.txt'}, page_content='LangChain is an open-source framework for'), Document(id='f27f263f-cd4e-432f-b111-b9913a4ea08f', metadata={'source': 'sample.txt'}, page_content='for building applications with large language'), Document(id='b1f92ffe-7bb0-4c77-8ac8-330cd00a81ca', metadata={'source': 'sample.txt'}, page_content='It allows developers to combine LLMs with')], 'answer': 'LangChain is an open-source framework designed for building applications that leverage large language models (LLMs). It enables developers to integrate these powerful language models with various data sources, tools, and other components to create applications that can understand and generate human-like text. LangChain supports the development of applications that can process natural language, answer questions, perform data retrieval, engage in conversation, and more, by allowing easy chaining of

In [13]:
qa.invoke({"input": "Who is Prashant Nair?"})

{'input': 'Who is Prashant Nair?',
 'context': [Document(id='dc85d7be-126a-4e82-9f54-c52ae2479173', metadata={'source': 'sample.txt'}, page_content='Prashant Nair is one of the best trainers of AI.'),
  Document(id='ffa61a02-1558-4732-b849-967f1822a4f9', metadata={'source': 'sample.txt'}, page_content='custom logic.'),
  Document(id='47b7d995-6695-43a4-bed3-d001c18b5707', metadata={'source': 'sample.txt'}, page_content='LLMs with external data sources, APIs, and custom')],
 'answer': 'Prashant Nair is recognized as one of the best trainers in the field of artificial intelligence (AI). He specializes in leveraging large language models (LLMs) and integrating them with external data sources, APIs, and custom logic to enhance their functionality and usability. His expertise helps practitioners understand how to effectively implement AI solutions that are tailored to specific needs and contexts.'}

In [14]:
from langchain_openai import ChatOpenAI
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

prompt_template = """You are a teacher teaching AI to students.

{context}

Question: {input}
Answer here:"""

PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "input"])

llm = ChatOpenAI(model="gpt-4o-mini")
stuff_chain = PROMPT | llm

qa = (
    {
        "context": (lambda x: x["input"]) | retriever,  # extract input -> retrieve docs
        "input": (lambda x: x["input"])                 # keep the original question
    }
    | stuff_chain
)

result = qa.invoke({"input": "What is Langchain??"})
print(result)


content='LangChain is an open-source framework designed for building applications that utilize large language models (LLMs). It provides developers with the tools to effectively combine LLMs with various data sources and applications, enabling the creation of more intelligent and interactive systems.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 169, 'total_tokens': 218, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a1ddba3226', 'id': 'chatcmpl-DGdYHdToqJ0IUzOmVI80FtE1ho2w5', 'finish_reason': 'stop', 'logprobs': None} id='run-43531a16-86d8-495c-851c-17ad50b687f5-0' usage_metadata={'input_tokens': 169, 'output_tokens': 49, 'total_tokens': 218, 'input_token_details': {'audio': 0, 'cache_read': 0}, '